# **Modified transformer training**

In [1]:
import sys, os

sys.path.append(os.path.abspath('..'))

### **Data loading**

In [2]:
import tensorflow as tf
import numpy as np
import tensorflow_datasets as tfds

ds = tfds.load('qm9', split='train')
ds = tfds.as_dataframe(ds)

features_to_drop = ['InChI', 'InChI_relaxed', 'SMILES_relaxed', 'tag', 'frequencies', 'charges', 'positions', 'Mulliken_charges', 'num_atoms', 'index']
ds = ds.drop(features_to_drop, axis=1)

ds_smiles = ds['SMILES'].astype('string')
ds_props = ds.drop(['SMILES'], axis=1)

2026-05-06 11:29:18.413539: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778059761.651847   15835 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13506 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-05-06 11:29:22.025472: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-05-06 11:30:13.407987: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### **Tokenizated data loading**

In [3]:
import json

with open('token_to_id.json', 'r') as f:
    token_to_id = json.load(f)
    
with open('id_to_token.json', 'r') as f:
    id_to_token = json.load(f) 

vocab_size = len(token_to_id)

### **Data Preprocessing**

In [ ]:
from utils_functions import smiles_to_selfies_tokens_ds, prepare_smiles_to_decoder_ds
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from utils_functions import get_props
import pandas as pd
import joblib

# Get additional properties
add_props = get_props(ds_smiles)
add_props = pd.DataFrame(add_props, columns=['QED', 'logP', 'Ctpsa', 'MW'])
ds_props = np.concat([ds_props, add_props], axis=1)

X_train_smiles, X_test_smiles, X_train_props, X_test_props = train_test_split(ds_smiles, ds_props, test_size=0.2, random_state=42)
X_train_smiles, X_valid_smiles, X_train_props, X_valid_props = train_test_split(X_train_smiles, X_train_props, test_size=0.2, random_state=42)

SEQ_LEN = 25    # Sequence length

# Encoding to selfies format
X_train_selfies, train_remove_idx = smiles_to_selfies_tokens_ds(X_train_smiles, token_to_id, SEQ_LEN, pad='[PAD]')
X_valid_selfies, valid_remove_idx = smiles_to_selfies_tokens_ds(X_valid_smiles, token_to_id, SEQ_LEN, pad='[PAD]')

X_train_smiles = np.delete(X_train_smiles, train_remove_idx, axis=0)
X_train_props = np.delete(X_train_props, train_remove_idx, axis=0)

X_valid_smiles = np.delete(X_valid_smiles, valid_remove_idx, axis=0)
X_valid_props = np.delete(X_valid_props, valid_remove_idx, axis=0)

# Tokenization smiles with time shift 
X_train_decoder_in, train_target = prepare_smiles_to_decoder_ds(X_train_smiles, token_to_id, SEQ_LEN, start='[START]', end='[END]', pad='[PAD]')
X_valid_dec_in, valid_target = prepare_smiles_to_decoder_ds(X_valid_smiles, token_to_id, SEQ_LEN, start='[START]', end='[END]', pad='[PAD]')

# Data scaling
std_sclr = StandardScaler()
X_train_props_std = std_sclr.fit_transform(X_train_props)
X_valid_props_std = std_sclr.transform(X_valid_props)

joblib.dump(std_sclr, 'StandardScaler.pkl')

Number of skipped smiles: 95
Number of skipped smiles: 22
Number of skipped smiles: 0
Number of skipped smiles: 0


['StandardScaler.pkl']

### **Hyperparameters**

In [ ]:
from transformer import TransformerNoEnc

# Model creation
num_layers = 2
d_model = 64
num_heads = 4
dff = 128
dropout = 0.1

transf_no_enc = TransformerNoEnc(
    num_layers=num_layers,
    d_model=d_model,
    num_heads=num_heads,
    dff=dff,
    seq_len=SEQ_LEN,
    dropout=dropout,
    vocab_size=vocab_size
)

## **Training**

In [6]:
from transformer import loss_fn

transf_no_enc.compile(
    loss=loss_fn,
    optimizer=tf.keras.optimizers.Adam(1e-3),
)

In [7]:
transf_no_enc.fit((X_train_props_std, X_train_decoder_in), train_target,
                  validation_data=((X_valid_props_std, X_valid_dec_in), valid_target),
                  epochs=10,
                  batch_size=32)

Epoch 1/10


/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'positional_encoding' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
2026-05-06 11:33:04.128112: I external/local_xla/xla/service/service.cc:163] XLA service 0x794214003b20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-05-06 11:33:04.128127: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Ti SUPER, Compute Capability 8.9
2026-05-06 11:33:04.333452: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-05-06 11:33:04.918548: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ig

  28/2614 ━━━━━━━━━━━━━━━━━━━━ 14s 6ms/step - loss: 2.2526 

I0000 00:00:1778060008.589236   15881 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


2609/2614 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.0124

2026-05-06 11:33:38.457537: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/loss_fn/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-06 11:33:39.272117: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-06 11:33:39.272207: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-06 11:33:39.272254: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently ma

2614/2614 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 1.0119

2026-05-06 11:34:04.907635: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/loss_fn/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-06 11:34:08.821263: W tensorflow/compiler/tf2xla/kernels/assert_op.cc:39] Ignoring Assert operator compile_loss/loss_fn/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
2026-05-06 11:34:08.955259: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-05-06 11:34:08.955377: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using th

2614/2614 ━━━━━━━━━━━━━━━━━━━━ 86s 20ms/step - loss: 0.7772 - val_loss: 0.6072
Epoch 2/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - loss: 0.5562 - val_loss: 0.5193
Epoch 3/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - loss: 0.5020 - val_loss: 0.4810
Epoch 4/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.4696 - val_loss: 0.4676
Epoch 5/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.4464 - val_loss: 0.4488
Epoch 6/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.4277 - val_loss: 0.4221
Epoch 7/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.4136 - val_loss: 0.4146
Epoch 8/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - loss: 0.4011 - val_loss: 0.4035
Epoch 9/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.3911 - val_loss: 0.3962
Epoch 10/10
2614/2614 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - loss: 0.3816 - val_loss: 0.3935


In [8]:
transf_no_enc.save_weights('../SavedWeights/corr_transformer_weights.weights.h5')